# Práctica — Clase 9: preparación de datos volumétricos

Completa i due esercizi. Ognuno ha celle con `# TODO:` e una soluzione nascosta sotto.

In [ ]:
import numpy as np
import tensorflow as tf

def generar_volumen_sintetico(num_cortes=40, alto=128, ancho=128, con_nodulo=True, semilla=None):
    rng = np.random.default_rng(semilla)
    volumen = rng.normal(loc=0.2, scale=0.05, size=(num_cortes, alto, ancho)).astype("float32")
    if con_nodulo:
        centro = (num_cortes // 2, alto // 2, ancho // 2)
        radio = 8
        z, y, x = np.ogrid[:num_cortes, :alto, :ancho]
        mascara_esfera = (z - centro[0]) ** 2 + (y - centro[1]) ** 2 + (x - centro[2]) ** 2 <= radio ** 2
        volumen[mascara_esfera] += 0.6
    return np.clip(volumen, 0, 1)

## Esercizio 1 — Completa il ridimensionamento in profondità

**Enunciato**: completa `redimensionar_volumen` affinché, dopo aver ridimensionato altezza e larghezza, trasponga il volume, ridimensioni la profondità alla dimensione target, e trasponga di nuovo all'ordine originale (profondità, altezza, larghezza, canale).

**Suggerimento**: `tf.transpose(volumen, perm=[2, 0, 1, 3])` sposta la profondità nella posizione dove `tf.image.resize` può trattarla come una dimensione spaziale.


In [ ]:
def redimensionar_volumen(volumen, tamano_destino=(64, 64, 32)):
    volumen = tf.expand_dims(volumen, axis=-1)
    volumen = tf.image.resize(volumen, tamano_destino[:2])
    # TODO: transpón el volumen con perm=[2, 0, 1, 3]
    # TODO: redimensiona a [tamano_destino[2], tamano_destino[0]]
    # TODO: transpón de vuelta con perm=[1, 2, 0, 3]
    return volumen

volumen = generar_volumen_sintetico(semilla=0)
volumen_redimensionado = redimensionar_volumen(volumen)
print("Forma esperada (64, 64, 32, 1):", volumen_redimensionado.shape)

<details>
<summary>💡 Vedi soluzione</summary>

```python
def redimensionar_volumen(volumen, tamano_destino=(64, 64, 32)):
    volumen = tf.expand_dims(volumen, axis=-1)
    volumen = tf.image.resize(volumen, tamano_destino[:2])
    volumen = tf.transpose(volumen, perm=[2, 0, 1, 3])
    volumen = tf.image.resize(volumen, [tamano_destino[2], tamano_destino[0]])
    volumen = tf.transpose(volumen, perm=[1, 2, 0, 3])
    return volumen
```
</details>

## Esercizio 2 — Costruisci il dataset etichettato

**Enunciato**: completa il ciclo affinché, ogni 2 pazienti generati, 1 abbia un nodulo (`con_nodulo=True`, etichetta 1) e l'altro no (etichetta 0), e aggiungi ogni volume ridimensionato e la sua etichetta alle liste corrispondenti.

**Suggerimento**: usa `i % 2 == 0` per alternare tra pazienti con e senza nodulo.


In [ ]:
volumenes = []
etiquetas = []

for i in range(20):
    # TODO: decide con_nodulo según si i es par o impar
    # TODO: genera el volumen, redimensiónalo, y añádelo a 'volumenes'
    # TODO: añade la etiqueta correcta (1 si tiene nódulo, 0 si no) a 'etiquetas'
    pass

X = np.array(volumenes)
y = np.array(etiquetas)
print("Forma de X:", X.shape)
print("Distribución de etiquetas:", np.bincount(y))

<details>
<summary>💡 Vedi soluzione</summary>

```python
volumenes = []
etiquetas = []

for i in range(20):
    con_nodulo = i % 2 == 0
    volumen = generar_volumen_sintetico(con_nodulo=con_nodulo, semilla=i)
    volumen = redimensionar_volumen(volumen)
    volumenes.append(volumen.numpy())
    etiquetas.append(1 if con_nodulo else 0)
```
</details>